# State Objective Training Target v1

"
        "Purpose: convert the failed `state_conditioned_objective_screen_v1` into an executable, per-date training/checkpoint-selection target instead of opening another blind PPO sweep.

"
        "Governance:
"
        "- canonical baseline: `v19reg_v22_budget_usage_bull_guard_v1`
"
        "- direct comparison baseline: `state_conditioned_objective_screen_v1` / `training_objective_handoff_v1`
"
        "- role: `diagnose`
"
        "- changed layer: `model_selection`, `reward`
"
        "- target: `>=30pp` relative return and `<=15%` MaxDD across walk-forward/state slices

"
        "External basis used for this loop: Qlib reproducible AI investment workflow, FinRL DRL trading environment/reward separation, and Probability of Backtest Overfitting walk-forward/OOS discipline.
"
        

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
for parent in [Path.cwd(), *Path.cwd().parents]:
    if (parent / 'src' / 'soxl_quant').exists():
        REPO_ROOT = parent
        break
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
from IPython.display import Markdown, display

from src.soxl_quant.app.state_objective_training_targets import (
    build_state_objective_training_targets,
    summarize_state_objective_training_targets,
)

EXP = 'state_objective_training_target_v1'
OUT_DIR = REPO_ROOT / 'runtime/model_selection/card' / EXP
SCREEN_DIR = REPO_ROOT / 'runtime/model_selection/card/state_conditioned_objective_screen_v1'
TEACHER_DIR = REPO_ROOT / 'runtime/reward/card/state_conditioned_rebound_risk_router_v1'
OUT_DIR.mkdir(parents=True, exist_ok=True)

screen_selection = json.loads((SCREEN_DIR / 'selection.json').read_text())
selected_candidate = screen_selection['summary']['selected_candidate']
selected_candidate


'state_conditioned_rebound_risk_router_v1'

In [2]:
daily_paths = {
    'strict_oos': pd.read_csv(TEACHER_DIR / 'selected_strict_oos_daily.csv'),
    'wf_2020': pd.read_csv(TEACHER_DIR / 'selected_wf_2020_daily.csv'),
    'wf_2022': pd.read_csv(TEACHER_DIR / 'selected_wf_2022_daily.csv'),
    'wf_2023_2024': pd.read_csv(TEACHER_DIR / 'selected_wf_2023_2024_daily.csv'),
}
window_metrics = pd.read_csv(SCREEN_DIR / 'window_metrics.csv')
cluster_metrics = pd.read_csv(SCREEN_DIR / 'cluster_metrics.csv')

targets = build_state_objective_training_targets(
    daily_paths,
    window_metrics=window_metrics,
    cluster_metrics=cluster_metrics,
    candidate=selected_candidate,
)
summary = summarize_state_objective_training_targets(targets)

targets_path = OUT_DIR / 'training_targets.csv'
window_summary_path = OUT_DIR / 'window_target_summary.csv'
summary_path = OUT_DIR / 'summary.json'

targets.to_csv(targets_path, index=False)
window_summary = pd.DataFrame(summary['window_summary'])
window_summary.to_csv(window_summary_path, index=False)
summary_path.write_text(json.dumps(summary, indent=2), encoding='utf-8')

summary


{'rows': 2322,
 'windows': ['strict_oos', 'wf_2020', 'wf_2023_2024', 'wf_2022'],
 'top_weighted_window': 'strict_oos',
 'max_objective_weight': 142.99498767638627,
 'rebound_rows': 94,
 'downtrend_rows': 1545,
 'mean_teacher_position': 0.15044766282786987,
 'mean_rebound_target_position': 0.15839454968397468,
 'mean_risk_target_position': 0.14438384406315388,
 'window_summary': [{'window': 'strict_oos',
   'rows': 1569,
   'mean_objective_weight': 12.837449743780887,
   'max_objective_weight': 142.99498767638627,
   'rebound_rows': 94,
   'downtrend_rows': 1001,
   'mean_teacher_position': 0.16141948655113716,
   'mean_rebound_target_position': 0.17318027130567845,
   'mean_risk_target_position': 0.15491864153266272},
  {'window': 'wf_2020',
   'rows': 252,
   'mean_objective_weight': 87.79081398524004,
   'max_objective_weight': 98.26236417611454,
   'rebound_rows': 0,
   'downtrend_rows': 118,
   'mean_teacher_position': 0.17697255042414278,
   'mean_rebound_target_position': 0.17697

In [3]:
display(Markdown('## Window Target Summary'))
display(window_summary)

display(Markdown('## Highest-Weight Target Dates'))
cols = [
    'datetime', 'window', 'cluster', 'teacher_position', 'rebound_target_position',
    'risk_target_position', 'objective_weight', 'walkforward_relative_weight',
    'downtrend_drawdown_weight', 'rebound_participation_weight',
    'window_relative_return', 'window_max_dd', 'rebound_state', 'risk_state',
]
display(targets.sort_values('objective_weight', ascending=False)[cols].head(20))


## Window Target Summary

,window,rows,mean_objective_weight,max_objective_weight,rebound_rows,downtrend_rows,mean_teacher_position,mean_rebound_target_position,mean_risk_target_position
0,strict_oos,1569,12.837450,142.994988,94,1001,0.161419,0.173180,0.154919
1,wf_2020,252,87.790814,98.262364,0,118,0.176973,0.176973,0.170263
2,wf_2023_2024,251,22.545804,30.391287,0,184,0.159488,0.159488,0.150765
3,wf_2022,250,1.000000,1.000000,0,242,0.045775,0.045775,0.045775


## Highest-Weight Target Dates

,datetime,window,cluster,teacher_position,rebound_target_position,risk_target_position,objective_weight,walkforward_relative_weight,downtrend_drawdown_weight,rebound_participation_weight,window_relative_return,window_max_dd,rebound_state,risk_state
1351,2025-05-19,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1355,2025-05-23,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1348,2025-05-14,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1349,2025-05-15,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1350,2025-05-16,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1352,2025-05-20,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1353,2025-05-21,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1354,2025-05-22,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1356,2025-05-27,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False
1345,2025-05-09,strict_oos,cluster_2025_high_vol_rebound,0.187500,0.350000,0.1875,142.994988,2.331228,5.364015,134.299746,26.891697,15.715202,False,False


In [4]:
selection = {
    'experiment': EXP,
    'decision': 'training_target_materialized_no_runtime_promotion',
    'promotion': 'blocked_training_target_only',
    'role': 'diagnose',
    'changed_layer': ['model_selection', 'reward'],
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': 'state_conditioned_objective_screen_v1',
    'teacher_candidate': selected_candidate,
    'target': {
        'relative_return_min_pp': 30.0,
        'max_dd_max_pct': 15.0,
    },
    'summary': {k: v for k, v in summary.items() if k != 'window_summary'},
    'artifacts': {
        'notebook': str((OUT_DIR / f'{EXP}.ipynb').relative_to(REPO_ROOT)),
        'training_targets': str(targets_path.relative_to(REPO_ROOT)),
        'window_target_summary': str(window_summary_path.relative_to(REPO_ROOT)),
        'summary': str(summary_path.relative_to(REPO_ROOT)),
        'selection': str((OUT_DIR / 'selection.json').relative_to(REPO_ROOT)),
    },
    'external_references': screen_selection.get('external_references', []),
    'conclusion': [
        'The failed objective screen has been converted into per-date training/checkpoint-selection targets.',
        'This is not runtime promotion evidence and does not satisfy the 30pp/15pct gate by itself.',
        'Next executable step is wiring this target table into behavior-cloning warm start, PPO reward shaping, or checkpoint scoring.',
        'The highest weights concentrate the next training signal on the worst walk-forward/rebound state dates instead of another broad blind sweep.',
    ],
}
(OUT_DIR / 'selection.json').write_text(json.dumps(selection, indent=2), encoding='utf-8')

display(Markdown(
    f"""## Conclusion\n\n"
        f"- decision: `{selection['decision']}`\n"
        f"- teacher candidate: `{selected_candidate}`\n"
        f"- rows: `{summary['rows']}` across `{len(summary['windows'])}` windows\n"
        f"- top weighted window: `{summary['top_weighted_window']}`\n"
        f"- rebound rows: `{summary['rebound_rows']}`; downtrend rows: `{summary['downtrend_rows']}`\n"
        f"- required terms: `{', '.join(summary['required_terms'])}`\n\n"
        "This materializes the training objective, but it is not a promoted model. The next loop should consume `training_targets.csv` inside checkpoint scoring / BC warm-start / PPO reward shaping and then rerun walk-forward validation."""
))
selection


## Conclusion

"
        f"- decision: `training_target_materialized_no_runtime_promotion`
"
        f"- teacher candidate: `state_conditioned_rebound_risk_router_v1`
"
        f"- rows: `2322` across `4` windows
"
        f"- top weighted window: `strict_oos`
"
        f"- rebound rows: `94`; downtrend rows: `1545`
"
        f"- required terms: `walkforward_relative_return_gap_penalty, walkforward_drawdown_gap_penalty, high_vol_rebound_participation_floor, downtrend_drawdown_state_cap`

"
        "This materializes the training objective, but it is not a promoted model. The next loop should consume `training_targets.csv` inside checkpoint scoring / BC warm-start / PPO reward shaping and then rerun walk-forward validation.

{'experiment': 'state_objective_training_target_v1',
 'decision': 'training_target_materialized_no_runtime_promotion',
 'promotion': 'blocked_training_target_only',
 'role': 'diagnose',
 'changed_layer': ['model_selection', 'reward'],
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': 'state_conditioned_objective_screen_v1',
 'teacher_candidate': 'state_conditioned_rebound_risk_router_v1',
 'target': {'relative_return_min_pp': 30.0, 'max_dd_max_pct': 15.0},
 'summary': {'rows': 2322,
  'windows': ['strict_oos', 'wf_2020', 'wf_2023_2024', 'wf_2022'],
  'top_weighted_window': 'strict_oos',
  'max_objective_weight': 142.99498767638627,
  'rebound_rows': 94,
  'downtrend_rows': 1545,
  'mean_teacher_position': 0.15044766282786987,
  'mean_rebound_target_position': 0.15839454968397468,
  'mean_risk_target_position': 0.14438384406315388,
  'required_terms': ['walkforward_relative_return_gap_penalty',
   'walkforward_drawdown_gap_penalty',
   'high_